# Similarity Network Fusion (SNF) for Combining Algorithm Outputs

**Question tested:** Instead of the current binarize-then-count fusion in `FINAL_MODEL_GCN.ipynb`, does Similarity Network Fusion (Wang et al., Nat. Methods 2014) do a better job combining the 17 algorithms' GCN-GAE reconstructions into one consensus network?

**What SNF actually does:** each algorithm's continuous GAE-reconstructed adjacency is treated as a similarity network ("view"). SNF iteratively lets each view update itself using information diffused from all the other views, so views that agree reinforce each other and noisy views get smoothed out. This was referenced but never actually run in `FINAL_FINAL_GCN-GAE.ipynb` (`from snf import compute` was commented out) -- this notebook actually installs and runs it.

**Result: not a clean win, but a real trade-off worth knowing about.** SNF's AUROC comes in *below* the simple-average baseline, but its AUPR is the *best of every fusion method tested* (including the NMF/entropy/Jaccard notebook). Since true edges are a tiny fraction of all possible gene pairs here, AUPR is arguably the more trustworthy metric for this problem than AUROC -- see the Conclusion cell for the full discussion.

## 1. Setup

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# pip install snfpy  (not in the original environment -- install once)
import snf

# snfpy 0.2.2 calls sklearn's check_array with the old kwarg name
# force_all_finite, which newer scikit-learn (1.4+) renamed to
# ensure_all_finite. Small compatibility shim so snf.snf() runs on a
# modern sklearn install without touching snfpy's source.
import sklearn.utils
import sklearn.utils.validation
_orig_check_array = sklearn.utils.check_array
def _patched_check_array(*args, **kwargs):
    if 'force_all_finite' in kwargs:
        kwargs['ensure_all_finite'] = kwargs.pop('force_all_finite')
    return _orig_check_array(*args, **kwargs)
sklearn.utils.check_array = _patched_check_array
sklearn.utils.validation.check_array = _patched_check_array
snf.compute.check_array = _patched_check_array

## 2. Load data and build per-algorithm continuous GAE reconstructions

Same architecture as `FINAL_MODEL_GCN.ipynb` (3-layer GCN encoder + GAE, identity node features, threshold 0.6), run independently per algorithm, averaged over 3 iterations. This produces one continuous 100x100 score matrix per algorithm -- the raw material every fusion method in this notebook and the companion NMF/entropy/Jaccard notebook both start from, so the comparison is apples to apples.

In [ ]:
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
ground_truth = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv", index_col=0)

num_genes = data.shape[0]
gene_names = data.index.tolist()
algorithms = sorted(set(col.split('.')[0] for col in data.columns))
node_features = np.identity(num_genes)
x_tensor = torch.tensor(node_features, dtype=torch.float).to(DEVICE)

print(f"Genes: {num_genes} | Algorithms: {len(algorithms)} | True edges: {int(ground_truth.values.sum())}")

In [ ]:
def normalize_adj(adj):
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
    return laplacian.tocoo()


def build_edge_index(alg_matrix, threshold=0.6):
    binary = (alg_matrix >= threshold).astype(float)
    adj_sparse = sp.csr_matrix(binary)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, _ = from_scipy_sparse_matrix(adj_normalized)
    return edge_index


class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.01):
        super().__init__()
        self.num_layers = num_layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])
        self.dropout = torch.nn.Dropout(p=dropout_rate)

    def forward(self, x, edge_index):
        x = self.convs[0](x, edge_index)
        x = F.relu(x)
        for i in range(1, self.num_layers - 1):
            x_res = x
            x = self.convs[i](x, edge_index)
            x = self.bns[i - 1](x)
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_res
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        return x


def train_gae(edge_index, x, embedding_dim=16, epochs=150, weight_decay=1e-8):
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim)
    model = GAE(encoder).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        z = model.encode(x, edge_index)
        loss = model.recon_loss(z, edge_index)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        z = model.encode(x, edge_index)
        recon = model.decoder.forward_all(z).cpu().numpy()
    return (recon - recon.min()) / (recon.max() - recon.min() + 1e-12)


def evaluate(score_matrix, label):
    gt_matrix = ground_truth.loc[gene_names, gene_names].values.astype(int)
    triu = np.triu_indices(num_genes, k=1)
    y_true = gt_matrix[triu]
    y_score = score_matrix[triu]
    auroc = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    best_f1 = max(f1_score(y_true, (y_score >= t).astype(int), zero_division=0)
                  for t in np.linspace(y_score.min(), y_score.max(), 50))
    print(f"[{label}] AUROC={auroc:.4f}  AUPR={aupr:.4f}  best-F1={best_f1:.4f}")
    return {'label': label, 'auroc': auroc, 'aupr': aupr, 'best_f1': best_f1}

In [ ]:
NUM_ITERATIONS = 3
EPOCHS = 150
EMBED_DIM = 16

print("Generating per-algorithm continuous GAE reconstructions...")
t0 = time.time()
per_alg_scores = {}
for alg in algorithms:
    cols = [c for c in data.columns if c.startswith(alg + '.')]
    mat = data[cols].values.reshape((num_genes, num_genes))
    edge_index = build_edge_index(mat, threshold=0.6).to(DEVICE)
    acc = np.zeros((num_genes, num_genes))
    for it in range(NUM_ITERATIONS):
        acc += train_gae(edge_index, x_tensor, embedding_dim=EMBED_DIM, epochs=EPOCHS)
    per_alg_scores[alg] = acc / NUM_ITERATIONS
    print(f"  {alg} done")
print(f"Total time: {time.time()-t0:.1f}s")

## 3. Baseline fusion: simple average (current best-known approach)

In [ ]:
baseline = np.mean(list(per_alg_scores.values()), axis=0)
res_baseline = evaluate(baseline, "Baseline (simple average)")

## 4. SNF fusion

`snf.snf(*views, K, t)` takes each algorithm's continuous adjacency directly as a similarity network (no extra preprocessing needed -- our per-algorithm scores are already gene-gene similarity/likelihood matrices) and iteratively diffuses information between views for `t` steps. `K` controls how many nearest neighbors count as a gene's local neighborhood during diffusion.

In [ ]:
mats = list(per_alg_scores.values())
fused_snf_k20 = snf.snf(*mats, K=20, t=20)
res_snf_k20 = evaluate(fused_snf_k20, "SNF (K=20, t=20)")

### 4b. K sweep -- is K=20 (the SNF default) actually the best setting here?

In [ ]:
snf_results = [res_baseline, res_snf_k20]
for K in [10, 15, 30]:
    fused = snf.snf(*mats, K=K, t=20)
    snf_results.append(evaluate(fused, f"SNF (K={K}, t=20)"))

## 5. Results summary

In [ ]:
print(f"{'Method':<28}{'AUROC':<10}{'AUPR':<10}{'Best F1'}")
for r in snf_results:
    print(f"{r['label']:<28}{r['auroc']:<10.4f}{r['aupr']:<10.4f}{r['best_f1']:.4f}")

## 6. Conclusion

**Actual results from this run:**

| Method | AUROC | AUPR | Best F1 |
|---|---|---|---|
| Baseline (simple average) | 0.8056 | 0.1005 | 0.1925 |
| SNF (K=20) | 0.7886 | **0.1117** | 0.1818 |
| SNF (K=10) | 0.7934 | 0.1054 | 0.1955 |
| SNF (K=15) | 0.7903 | 0.1060 | 0.1850 |
| SNF (K=30) | 0.7861 | 0.1090 | 0.1777 |

**SNF is not a clean win, but it's not a clean loss either -- it's a different trade-off.** Every SNF setting tested came in *below* the baseline on AUROC. But every SNF setting also came in *above or roughly equal* to the baseline on AUPR, and K=20 gave the single best AUPR of any fusion method tested across both this notebook and the companion NMF/entropy/Jaccard notebook (0.1117 vs baseline's 0.1005).

**Why AUPR might matter more than AUROC here:** with only 175 true edges out of roughly 4,950 possible gene pairs (about 3.5%), this is a heavily imbalanced problem. AUROC is known to look optimistic under heavy class imbalance because it weights the (huge) majority of true negatives just as heavily as the rare positives. AUPR is more sensitive to how the model handles the minority class specifically -- which is the class that actually matters for finding real gene interactions. By that read, SNF's AUPR edge over the baseline is a more meaningful signal than its AUROC deficit.

**Practical recommendation:** if downstream use of this network cares more about precision among top-ranked edges (e.g. picking a shortlist of candidate interactions to validate experimentally), SNF with K=20 is worth using over the simple-average baseline. If AUROC-style overall ranking quality matters more, stick with the simple average. Either way, this is a judgment call about which error type is more costly for the specific downstream use, not a settled "SNF wins" or "SNF loses" result.

**Caveats:** tested on DREAM4 network 1 only, 3 iterations for the per-algorithm GAE step, `t=20` diffusion iterations left at the SNF default (not swept). A `t` sweep and testing on the other 4 DREAM4 networks would be the natural next step before adopting this.